In [0]:
%sql
-- Intenção: provar particionamento, Z-Order e deletion vectors na Silver
DESCRIBE DETAIL lakehouse_cnes.silver.cnes_silver;

-- Conclusão esperada: partitionColumns = [ano, mes]; properties com delta.enableDeletionVectors e pipelines.autoOptimize.zOrderCols

In [0]:
%sql
-- Intenção: provar o Liquid Clustering na Gold (materialized view é registrada como view no UC)
DESCRIBE TABLE EXTENDED lakehouse_cnes.gold.fato_capacidade_municipio;

-- Conclusão esperada: seção de informações da tabela com o clustering por id_municipio

In [0]:
%sql
-- Intenção: protocolo de transação - cada execução do pipeline gera uma versão no log da tabela
DESCRIBE HISTORY lakehouse_cnes.bronze.cnes_estabelecimento_bronze;

-- Conclusão esperada: versões de STREAMING UPDATE, com numOutputRows 37.226 (1ª carga) e 74.199 (carga incremental)

In [0]:

%sql
-- Intenção: criar uma tabela Delta comum (fora do pipeline) com a competência 2026/02 para as operações de escrita
CREATE OR REPLACE TABLE lakehouse_cnes.silver.cnes_auditoria_delta
COMMENT 'Cópia da Silver 2026/02 para demonstração de UPDATE, DELETE, Time Travel, RESTORE, OPTIMIZE e VACUUM'
TBLPROPERTIES (
    'delta.enableDeletionVectors' = 'true'
)
AS
SELECT *
FROM lakehouse_cnes.silver.cnes_silver
WHERE ano = 2026
  AND mes = 2;

-- Conclusão esperada: versão 0 da tabela, com 36.717 registros

In [0]:
%sql
-- Intenção: estado de referência antes do incidente
SELECT
    COUNT(*) AS estabelecimentos,
    SUM(quantidade_leito_clinico) AS leitos_clinicos
FROM lakehouse_cnes.silver.cnes_auditoria_delta;

-- Conclusão esperada: 36.717 estabelecimentos e o total de leitos clínicos da competência

In [0]:
%sql
-- Intenção: simular o incidente 1 - atualização indevida que zera os leitos clínicos do município do Rio de Janeiro
UPDATE lakehouse_cnes.silver.cnes_auditoria_delta
SET quantidade_leito_clinico = 0
WHERE id_municipio = '3304557';

-- Conclusão esperada: versão 1 da tabela

In [0]:
%sql
-- Intenção: simular o incidente 2 - exclusão indevida dos consultórios isolados (tipo_unidade 22)
DELETE FROM lakehouse_cnes.silver.cnes_auditoria_delta
WHERE tipo_unidade = '22';

-- Conclusão esperada: versão 2 da tabela; exclusão feita com deletion vectors, sem reescrever os arquivos

In [0]:
%sql
-- Intenção: auditoria - o log de transações registra quem fez o quê, quando e com quais métricas
DESCRIBE HISTORY lakehouse_cnes.silver.cnes_auditoria_delta;

-- Conclusão esperada: versões 0 (CREATE), 1 (UPDATE) e 2 (DELETE); no DELETE, numDeletionVectorsAdded > 0

In [0]:
%sql
-- Intenção: Time Travel - comparar o estado atual com o estado anterior ao incidente
SELECT 'versao_0_antes_do_incidente' AS estado, COUNT(*) AS estabelecimentos, SUM(quantidade_leito_clinico) AS leitos_clinicos
FROM lakehouse_cnes.silver.cnes_auditoria_delta VERSION AS OF 0
UNION ALL
SELECT 'versao_atual_apos_incidente', COUNT(*), SUM(quantidade_leito_clinico)
FROM lakehouse_cnes.silver.cnes_auditoria_delta;

-- Conclusão esperada: a versão atual tem menos estabelecimentos e menos leitos que a versão 0

In [0]:
%sql
-- Intenção: recuperar o estado anterior ao incidente
RESTORE TABLE lakehouse_cnes.silver.cnes_auditoria_delta TO VERSION AS OF 0;

-- Conclusão esperada: versão 3 (RESTORE), com conteúdo idêntico à versão 0

In [0]:
%sql
-- Intenção: confirmar a recuperação
SELECT
    COUNT(*) AS estabelecimentos,
    SUM(quantidade_leito_clinico) AS leitos_clinicos
FROM lakehouse_cnes.silver.cnes_auditoria_delta;

-- Conclusão esperada: mesmos valores do estado de referência (36.717 estabelecimentos)

In [0]:
%sql
-- Intenção: compactação e ordenação Z-Order dos arquivos da tabela de auditoria
OPTIMIZE lakehouse_cnes.silver.cnes_auditoria_delta
ZORDER BY (id_estabelecimento_cnes, id_municipio);

-- Conclusão esperada: métricas com numFilesAdded e numFilesRemoved

In [0]:
%sql
-- Intenção: listar os arquivos que seriam removidos pela limpeza (simulação, sem apagar nada)
VACUUM lakehouse_cnes.silver.cnes_auditoria_delta RETAIN 168 HOURS DRY RUN;

-- Conclusão esperada: lista vazia ou poucos arquivos, pois a retenção de 7 dias protege as versões recentes do Time Travel

In [0]:
%sql
-- Intenção: simular fragmentação - três cargas pequenas e separadas (cada INSERT gera um arquivo novo)
INSERT INTO lakehouse_cnes.silver.cnes_auditoria_delta
SELECT * FROM lakehouse_cnes.silver.cnes_silver WHERE ano = 2026 AND mes = 1 AND id_municipio = '3303302';

INSERT INTO lakehouse_cnes.silver.cnes_auditoria_delta
SELECT * FROM lakehouse_cnes.silver.cnes_silver WHERE ano = 2026 AND mes = 1 AND id_municipio = '3301009';

INSERT INTO lakehouse_cnes.silver.cnes_auditoria_delta
SELECT * FROM lakehouse_cnes.silver.cnes_silver WHERE ano = 2026 AND mes = 1 AND id_municipio = '3301702';

In [0]:
%sql
-- Intenção: número de arquivos antes da otimização
DESCRIBE DETAIL lakehouse_cnes.silver.cnes_auditoria_delta;

-- Conclusão esperada: numFiles = 4 (o original + um por INSERT)

In [0]:
%sql
-- Intenção: compactar os arquivos pequenos e ordenar por Z-Order
OPTIMIZE lakehouse_cnes.silver.cnes_auditoria_delta
ZORDER BY (id_estabelecimento_cnes, id_municipio);

-- Conclusão esperada: numFilesRemoved = 4, numFilesAdded = 1

In [0]:
%sql
-- Intenção: número de arquivos depois da otimização
DESCRIBE DETAIL lakehouse_cnes.silver.cnes_auditoria_delta;

-- Conclusão esperada: numFiles = 1